# **DINO enhanced visualisation**

## **1. Importing**

In [8]:
import torch
import numpy as np
import torchvision
from transformers import AutoImageProcessor, AutoModel
from torch.utils.data import DataLoader
import h5py
import random
import kagglehub
import os
from torchvision.datasets import ImageFolder
from torchvision import transforms
import matplotlib.pyplot as plt

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### **1.2 DINO FLAGSHIP MODEL 40 LAYERS WOHOO**

In [10]:
model_name = "facebook/dinov3-vit7b16-pretrain-lvd1689m"

print(f"Loading flagship model: {model_name}")
print(f"Targeting Architecture: 40 Layers, 4096 Embedding Dim")

try:
    # 3. Load Model with Memory Optimization
    model = AutoModel.from_pretrained(
        model_name,
        torch_dtype=dtype,
        low_cpu_mem_usage=True
    ).to(device).eval()

    # 4. Load Processor
    processor = AutoImageProcessor.from_pretrained(model_name)
    transform = lambda img: processor(images=img, return_tensors="pt")['pixel_values'].squeeze()

    # 5. Verify Layer Count
    # Depending on the specific Hugging Face implementation, blocks might be in 'model.layers'
    if hasattr(model, "vision_model"):
        num_layers = len(model.vision_model.encoder.layers)
    else:
        # Fallback for flat architectures
        num_layers = len(model.layer)

    print(f"Success! Model loaded with {num_layers} layers.")

except Exception as e:
    print(f"Error loading model: {e}")
    print("Tip: If you run out of memory (OOM), try 'dinov3-vitl16' (Large, 24 layers) instead.")

Loading flagship model: facebook/dinov3-vit7b16-pretrain-lvd1689m
Targeting Architecture: 40 Layers, 4096 Embedding Dim


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Success! Model loaded with 40 layers.


## **2. Dataset**

### **2.1 CIFAR**

In [11]:
dataset_cifar10 = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
loader_cifar10 = DataLoader(dataset_cifar10, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of CIFAR10 samples (OOD - CIFAR-10): {len(dataset_cifar10)}")

Amount of CIFAR10 samples (OOD - CIFAR-10): 10000


In [12]:
dataset_cifar100 = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
loader_cifar100 = DataLoader(dataset_cifar100, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of CIFAR100 samples (OOD - CIFAR-100): {len(dataset_cifar100)}")

Amount of CIFAR100 samples (OOD - CIFAR-100): 10000


### **2.2 SVHN**

In [13]:
dataset_svhn = torchvision.datasets.SVHN(root='./data', split='test', download=True, transform=transform)
loader_svhn = DataLoader(dataset_svhn, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of SVHN samples (Test set): {len(dataset_svhn)}")

Amount of SVHN samples (Test set): 26032


### **2.3 FOOD101**

In [14]:
dataset_food101 = torchvision.datasets.Food101(root='./data', split='test', download=True, transform=transform)
loader_food101 = DataLoader(dataset_food101, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of Food101 samples (test set): {len(dataset_food101)}")

100%|██████████| 5.00G/5.00G [02:52<00:00, 29.0MB/s]


Amount of Food101 samples (test set): 25250


### **2.4 Pokémon**

In [15]:
path = kagglehub.dataset_download("thedagger/pokemon-generation-one")

print("Path to dataset files:", path)

100%|██████████| 2.15G/2.15G [00:13<00:00, 171MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/thedagger/pokemon-generation-one/versions/1


In [16]:
transform_kaggle = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

image_root_path = os.path.join(path, 'dataset')

dataset_pokemon_gen1 = ImageFolder(
    root=image_root_path,
    transform=transform_kaggle
)

loader_pokemon_gen1 = DataLoader(
    dataset_pokemon_gen1,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

print(f"\n✅ Success! Loaded {len(dataset_pokemon_gen1)} images.")
print(f"Found classes: {dataset_pokemon_gen1.classes[:5]}...")


✅ Success! Loaded 21314 images.
Found classes: ['Abra', 'Aerodactyl', 'Alakazam', 'Arbok', 'Arcanine']...


## **3. Video Generation**

In [17]:
MODEL_NAME = "facebook/dinov3-vit7b16-pretrain-lvd1689m"
IMAGE_PATH = "your_high_res_image.jpg"  # Replace with your image
OUTPUT_VIDEO = "dinov3_attention_scan.mp4"
PATCH_SIZE = 16  # Standard patch size for ViT models
BIG_PATCH_SIZE = 2  # Combine 2x2 standard patches into one "Big Patch"
STRIDE = 1  # Step size for sliding the Big Patch (in units of standard patches)
FPS = 10  # Frames per second for the video

### **3.1 Implementation**

In [18]:
def get_attention_from_model(model, processor, image, device):
    """
    Runs the model and extracts the attention map from the last layer.
    Replicates logic from m6 notebook but adapted for the 7B model structure.
    """
    # Prepare image without aggressive downscaling if possible,
    # but keep within GPU memory limits.
    # For DINOv3 7B, we might need to tile or be careful with resolution.
    # Here we trust the processor but ensure return_tensors="pt".
    inputs = processor(images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Forward pass with output_attentions=True
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    # Get attention from the last layer
    # Shape: (batch_size, num_heads, num_tokens, num_tokens)
    last_layer_attn = outputs.attentions[-1]

    # Average attention across heads [0] is batch index
    # Shape: (num_tokens, num_tokens)
    avg_attn = last_layer_attn[0].mean(dim=0)

    # Remove CLS token (index 0) to focus on patch-to-patch attention
    # Shape: (num_patches, num_patches)
    patch_attn = avg_attn[1:, 1:]

    return patch_attn

In [19]:
def create_video_from_attention(image_path, model_name, output_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Loading model {model_name} on {device}...")

    # Load model and processor
    # Using float16 for 7B model to save memory
    model = AutoModel.from_pretrained(model_name, torch_dtype=torch.float16, low_cpu_mem_usage=True).to(device).eval()
    processor = AutoImageProcessor.from_pretrained(model_name)

    # Load and Prepare Image
    raw_image = Image.open(image_path).convert("RGB")

    # Calculate grid dimensions
    # Ensure dimensions are multiples of patch size
    w, h = raw_image.size
    w = (w // PATCH_SIZE) * PATCH_SIZE
    h = (h // PATCH_SIZE) * PATCH_SIZE
    resized_image = raw_image.resize((w, h))

    grid_w = w // PATCH_SIZE
    grid_h = h // PATCH_SIZE
    num_patches = grid_w * grid_h

    print(f"Image resized to {w}x{h} (Grid: {grid_w}x{grid_h} patches)")

    # Get Attention Matrix
    print("Extracting attention maps...")
    attn_matrix = get_attention_from_model(model, processor, resized_image, device)

    # Initialize Video Writer
    # Side-by-side: Original with box + Heatmap
    video_w = w * 2
    video_h = h
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video = cv2.VideoWriter(output_path, fourcc, FPS, (video_w, video_h))

    print("Generating frames...")

    # Iterate over the grid with the defined "Big Patch" window
    # We iterate in steps of STRIDE
    for y in range(0, grid_h - BIG_PATCH_SIZE + 1, STRIDE):
        for x in range(0, grid_w - BIG_PATCH_SIZE + 1, STRIDE):

            # 1. Identify indices of all small patches inside the current "Big Patch"
            indices = []
            for dy in range(BIG_PATCH_SIZE):
                for dx in range(BIG_PATCH_SIZE):
                    idx = (y + dy) * grid_w + (x + dx)
                    indices.append(idx)

            # 2. Aggregate attention:
            # Take the rows corresponding to these indices from the matrix
            # and sum them up (or max) to see what this region attends to.
            # attn_matrix shape: (num_patches, num_patches)
            # We extract rows [indices, :] -> Shape (N_small_patches, num_patches)
            # Then sum across dimension 0 -> Shape (num_patches)
            aggregated_attn = attn_matrix[indices, :].sum(dim=0)

            # 3. Reshape to 2D grid
            attn_map_2d = aggregated_attn.view(grid_h, grid_w).float().cpu().numpy()

            # 4. Create Heatmap Image
            # Normalize to 0-255
            attn_map_2d = (attn_map_2d - attn_map_2d.min()) / (attn_map_2d.max() - attn_map_2d.min() + 1e-6)
            attn_map_2d = (attn_map_2d * 255).astype(np.uint8)

            # Resize heatmap to original image size
            heatmap_resized = cv2.resize(attn_map_2d, (w, h), interpolation=cv2.INTER_NEAREST)
            heatmap_colored = cv2.applyColorMap(heatmap_resized, cv2.COLORMAP_JET)

            # 5. Create Reference Image with Box
            ref_img_cv = np.array(resized_image)
            # Convert RGB (PIL) to BGR (OpenCV)
            ref_img_cv = cv2.cvtColor(ref_img_cv, cv2.COLOR_RGB2BGR)

            # Draw rectangle around the "Big Patch"
            # Coordinates in pixels
            x1 = x * PATCH_SIZE
            y1 = y * PATCH_SIZE
            x2 = (x + BIG_PATCH_SIZE) * PATCH_SIZE
            y2 = (y + BIG_PATCH_SIZE) * PATCH_SIZE
            cv2.rectangle(ref_img_cv, (x1, y1), (x2, y2), (0, 255, 0), 3) # Green box

            # 6. Combine and Write
            # Add heatmap as overlay on the right side? Or separate?
            # Let's put them side-by-side
            combined_frame = np.hstack((ref_img_cv, heatmap_colored))
            video.write(combined_frame)

    video.release()
    print(f"Video saved to {output_path}")